# সহজ ভাষায় Notebook Guide

এই notebook-এ TensorFlow/Keras-এর concept এবং project code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, explanation Bangla-তে থাকবে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন।
2. Run করার আগে expected shape বা value লিখে নিন।
3. Training loss-এর পাশাপাশি validation loss দেখুন।
4. Scaled prediction original 0–100 scale-এ ফেরত আনার step বাদ দেবেন না।
5. Error হলে Python environment, project path, input shape এবং dtype check করুন।

> Notebook source intentionally unexecuted রাখা হয়েছে। Project requirements install করা environment/kernel select করে run করুন।

# Class 14 — Tensors, TensorFlow, and Keras
## Hands-On Lab: Industrial Equipment Success Score

### Learning Objectives

- Tensor-এর rank, shape এবং dtype inspect করা
- tf.GradientTape দিয়ে automatic differentiation দেখা
- ১৩টি raw predictor কীভাবে ২৬টি encoded input column হয় তা trace করা
- Keras Sequential regression model build, train এবং evaluate করা
- Prediction original score scale-এ ফেরত আনা
- .keras save/load round-trip validate করা

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

candidates = [
    Path.cwd().parent / "Class 2 Project",
    Path.cwd() / "week 7" / "Class 2 Project",
    Path.cwd(),
]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src").is_dir() and (p / "config.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Week 7/Class 2 Project খুঁজে পাওয়া যায়নি। Notebook repo root বা lecture folder থেকে খুলুন।")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from src.data.generator import generate_equipment_data
from src.data.preprocessor import DataPreprocessor
from src.models.builder import build_model

tf.keras.utils.set_random_seed(42)
np.set_printoptions(precision=3, suppress=True)
print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## Part 1: Tensor Shape, Rank, and Dtype

Tensor হলো typed, multi-dimensional array—batch size, feature width এবং data type model contract-এর অংশ।

In [ ]:
scalar = tf.constant(72.5, dtype=tf.float32)
vector = tf.constant([72.5, 68.0, 75.3], dtype=tf.float32)
matrix = tf.constant([[65.0, 2.1, 8.8], [78.0, 4.2, 9.5]], dtype=tf.float32)

for name, tensor in {"scalar": scalar, "vector": vector, "matrix": matrix}.items():
    print(f"{name:>6} | rank={tf.rank(tensor).numpy()} | shape={tensor.shape} | dtype={tensor.dtype.name}")

assert matrix.shape == (2, 3)
print("\nMatrix mean by feature:", tf.reduce_mean(matrix, axis=0).numpy())

### Shape Interpretation

- matrix-এর axis 0 হলো sample/equipment row।
- axis 1 হলো feature column।
- Dense layer সাধারণত shape (batch_size, input_features) আশা করে।
- Wrong rank বা wrong width হলে training-এর আগে shape error হওয়া ভালো—silent wrong result-এর চেয়ে এটি safer।

## Part 2: Automatic Differentiation

Backpropagation-এর core idea হলো loss কোন parameter-এর দিকে কত দ্রুত বদলায় তা gradient দিয়ে বের করা।

In [ ]:
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2

dy_dx = tape.gradient(y, x)
print(f"x={x.numpy():.1f}, y=x^2={y.numpy():.1f}, dy/dx={dy_dx.numpy():.1f}")
assert np.isclose(dy_dx.numpy(), 6.0)

# একটি tiny gradient-descent step
learning_rate = 0.1
x.assign_sub(learning_rate * dy_dx)
print("Updated x:", x.numpy())

## Part 3: Raw Data থেকে Model Tensor

Project generator reproducible synthetic data তৈরি করে। DataPreprocessor numeric columns scale করে, categorical columns one-hot encode করে এবং target scale করে।

In [ ]:
df = generate_equipment_data(n_samples=1200, seed=42)
raw_predictors = [c for c in df.columns if c not in {"equipment_id", "success_score"}]

print("Dataset shape:", df.shape)
print("Raw predictor count:", len(raw_predictors))
print("Target range:", (df["success_score"].min(), df["success_score"].max()))
display(df.head(3))

assert len(raw_predictors) == 13
assert df["equipment_id"].is_unique
assert df["success_score"].between(0, 100).all()

In [ ]:
preprocessor = DataPreprocessor()
X, y_scaled = preprocessor.fit_transform(df)

print("Encoded X shape:", X.shape)
print("Scaled y shape:", y_scaled.shape)
print("Numeric columns:", len(preprocessor.numeric_features))
print("Encoded columns:", len(preprocessor.feature_names_after_transform))
print("First 8 encoded names:", preprocessor.feature_names_after_transform[:8])

assert X.shape == (1200, 26)
assert y_scaled.shape == (1200,)
assert np.isfinite(X).all() and np.isfinite(y_scaled).all()

### কেন ১৩ থেকে ২৬?

১০টি numeric column একই থাকে। Equipment type-এর ৬, manufacturer-এর ৬ এবং facility-এর ৪ category one-hot column দেয়: ১০ + ৬ + ৬ + ৪ = ২৬। Category catalog বদলালে width-ও বদলাতে পারে, তাই trainer processed array থেকে input dimension infer করে।

## Part 4: Build and Train a Small Keras Model

Lab দ্রুত রাখার জন্য project default-এর চেয়ে ছোট network এবং কম epoch ব্যবহার করা হচ্ছে। Test split model selection-এ ব্যবহার হবে না।

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_scaled, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

model = build_model(
    input_dim=X_train.shape[1],
    hidden_units=[64, 32],
    dropout_rate=0.15,
    learning_rate=0.001,
)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Input shape:", model.input_shape, "Output shape:", model.output_shape)
model.summary()
assert model.input_shape == (None, 26)
assert model.output_shape == (None, 1)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
    verbose=0,
)

print("Epochs trained:", len(history.history["loss"]))
print("Final train loss:", round(history.history["loss"][-1], 4))
print("Final validation loss:", round(history.history["val_loss"][-1], 4))

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(9, 4))
plt.plot(epochs, history.history["loss"], label="Train loss")
plt.plot(epochs, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE on scaled target")
plt.title("Learning Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Learning Curve কীভাবে পড়বেন?

- দুই loss কমলে model useful pattern শিখছে।
- Train loss কমলেও validation loss বাড়লে overfitting signal।
- দুই loss-ই বেশি ও flat হলে model/data/learning-rate underfitting বা optimization problem হতে পারে।
- এখানে loss scaled target-এর ওপর; original score-unit error পরের cell-এ হিসাব হবে।

## Part 5: Evaluate on the Original 0–100 Scale

In [ ]:
pred_scaled = model.predict(X_test, verbose=0).reshape(-1)
pred_score = preprocessor.inverse_transform_target(pred_scaled)
true_score = preprocessor.inverse_transform_target(y_test)

mae = mean_absolute_error(true_score, pred_score)
rmse = np.sqrt(mean_squared_error(true_score, pred_score))
baseline = np.full_like(true_score, true_score.mean())
baseline_mae = mean_absolute_error(true_score, baseline)

print(f"Model MAE:    {mae:.2f} score points")
print(f"Model RMSE:   {rmse:.2f} score points")
print(f"Baseline MAE: {baseline_mae:.2f} score points")
assert np.isfinite(pred_score).all()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_score, pred_score, alpha=0.5)
lims = [min(true_score.min(), pred_score.min()), max(true_score.max(), pred_score.max())]
axes[0].plot(lims, lims, "r--", label="Perfect prediction")
axes[0].set(xlabel="Actual score", ylabel="Predicted score", title="Actual vs Predicted")
axes[0].legend()

residuals = true_score - pred_score
axes[1].hist(residuals, bins=25, color="teal", alpha=0.75)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(xlabel="Actual - predicted", ylabel="Count", title="Residual Distribution")
plt.tight_layout()
plt.show()

## Part 6: Save/Load Round-Trip

Current Keras format পুরো model archive save করে। Reload-এর পরে একই input-এর prediction materially identical হওয়া উচিত।

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    model_path = Path(tmp_dir) / "equipment_lab.keras"
    model.save(model_path)
    restored_model = tf.keras.models.load_model(model_path)
    restored_pred = restored_model.predict(X_test[:10], verbose=0)

original_pred = model.predict(X_test[:10], verbose=0)
max_difference = np.max(np.abs(original_pred - restored_pred))
print("Maximum prediction difference after reload:", max_difference)
assert np.allclose(original_pred, restored_pred, atol=1e-6)

## Project Handoff

Full project run করতে Class 2 Project directory থেকে:

    python run_pipeline.py --epochs 5
    python -m streamlit run app/main.py

### Summary Checklist

- [ ] Tensor rank, shape ও dtype explain করতে পারি।
- [ ] GradientTape কেন backpropagation-এর foundation বুঝি।
- [ ] ১৩ raw predictor থেকে ২৬ encoded input কেন হয় বুঝি।
- [ ] Test set model selection-এ ব্যবহার করিনি।
- [ ] Scaled prediction original score scale-এ ফিরিয়েছি।
- [ ] Model baseline-এর তুলনায় useful কি না check করেছি।
- [ ] Save/load prediction parity verify করেছি।

### Final Reflection

নিজের ভাষায় লিখুন: preprocessing artifact model-এর মতোই গুরুত্বপূর্ণ কেন, validation loss কী signal দেয়, এবং synthetic data-তে ভালো result real factory-তে deployment-এর জন্য যথেষ্ট নয় কেন?